# Lesson 4.2: Geoparsing in Python


## A big thank you!
This lesson is made possible by the [geoparser](https://github.com/dguzh/geoparser) library created by Diego Gomes. 

## Overview

In this lesson, we'll take text data we scraped from Reddit and:

1. **Extract Locations**: Use a sophisticated geoparser to find and resolve geographic references
3. **Visualize on Maps**: Create interactive maps showing retrieved locations


---

In [1]:
# Import all required libraries
try:
    from geoparser import Geoparser
    from tqdm.notebook import tqdm
    import pandas as pd
    import plotly.express as px
    import mapclassify as mc
    import warnings
    
    # Suppress warnings for cleaner output
    warnings.simplefilter(action='ignore', category=FutureWarning)
    
    print("✅ All libraries imported successfully!")
    
except ImportError as e:
    print(f"❌ Missing library: {e}")
    print("Please run the installation notebook first: lesson_5_0_installation_setup.ipynb")

✅ All libraries imported successfully!


## Part 1: Initialize the Geoparser System

The geoparser is a sophisticated tool that can identify place names in text and resolve them to actual geographic coordinates. Here **resolve** means figure out the coordinates. The fancy word for this is actually **toponym disambiguation**. For example, we found Harrisonburg in our text, but we don't know necessarily which Harrisonburg. There is also a Harrisonburg in Louisiana. During toponym resolution the geoparser makes an educated guess as to which location is the right one. This can be based on a number of variables including the surrounding context, other places in the corpus, and even the weighting of the town size in a gazetteer. Harrisonburg, LA only has a few hundred residents, it is less likely someone is talking about that place.

Google Maps also uses this technique to figure out what location you might be trying to find when you type it in on maps. Note that when you search for "London", London, UK is the first hit even though London,KY is closer.

### Step 1.1: Initialize the Geoparser

We'll create a geoparser with optimized settings for accuracy:

In [2]:
try:
    print("Initializing geoparser... (this may take a minute)")
    geo = Geoparser(
        spacy_model='en_core_web_trf',                    # Advanced language model
        transformer_model='dguzh/geo-all-distilroberta-v1', # Geographic transformer
        gazetteer='geonames'                              # Geographic database
    )
    print("✅ Geoparser initialized successfully!")
    
except Exception as e:
    print(f"❌ Error initializing geoparser: {e}")
    print("Make sure you ran the installation notebook first!")

Initializing geoparser... (this may take a minute)
✅ Geoparser initialized successfully!


**What these parameters do:**
- `spacy_model`: Advanced language processing for accurate text understanding
- `transformer_model`: Specialized AI model trained to recognize geographic references  
- `gazetteer`: Database containing millions of place names and their coordinates

### Step 1.2: Test the Geoparser

Let's test the geoparser with some sample sentences. Try changing the text below to include places you know:

In [3]:
# Test with sample sentences - feel free to modify these!
test_sentences = [
    "I traveled from New York to Richmond, Virginia last summer.",
    "The battle took place near Harrisonburg in the Shenandoah Valley.",
    "London and Paris are popular European destinations."
]

try:
    docs = geo.parse(test_sentences)
    print(f"✅ Successfully parsed {len(docs)} sentences!")
    
except Exception as e:
    print(f"❌ Error during parsing: {e}")
    print("Try restarting the kernel and running from the beginning.")

Toponym Recognition...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Toponym Resolution...


Batches:   0%|          | 0/66 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Successfully parsed 3 sentences!


### Step 1.3: Examine the Results

Let's see what locations the geoparser found. Each "toponym" is a place name with detailed geographic information:

In [4]:
print("🗺️  LOCATIONS FOUND:")
print("=" * 50)

for i, doc in enumerate(docs):
    print(f"\nSentence {i+1}: \"{test_sentences[i]}\"")
    
    if doc.toponyms:
        for toponym in doc.toponyms:
            print(f"  📍 Found: {toponym}")
    else:
        print("  ❌ No locations found in this sentence")
        
print("\n" + "=" * 50)

🗺️  LOCATIONS FOUND:

Sentence 1: "I traveled from New York to Richmond, Virginia last summer."
  📍 Found: New York
  📍 Found: Richmond
  📍 Found: Virginia

Sentence 2: "The battle took place near Harrisonburg in the Shenandoah Valley."
  📍 Found: Harrisonburg
  📍 Found: the Shenandoah Valley

Sentence 3: "London and Paris are popular European destinations."
  📍 Found: London
  📍 Found: Paris



### Understanding the Data Structure

Each toponym contains detailed geographic information. Here's what a complete location record looks like:

```python
{
    'geonameid': 2867714,
    'name': 'Munich',
    'latitude': 48.13743,
    'longitude': 11.57549,
    'country_name': 'Germany',
    'admin1_name': 'Bavaria',        # State/Province
    'admin2_name': 'Upper Bavaria',  # County/Region
    'feature_name': 'seat of a first-order administrative division',
    'population': 1260391
}
```

We can access specific pieces of information using `.location['key_name']`:

In [5]:
# Extract specific geographic information
print("📍 DETAILED LOCATION DATA:")
print("=" * 60)

for i, doc in enumerate(docs):
    print(f"\nSentence {i+1}: \"{test_sentences[i]}\"")
    
    for toponym in doc.toponyms:
        if toponym.location:
            name = toponym.location['name']
            lat = toponym.location['latitude']
            lon = toponym.location['longitude']
            country = toponym.location.get('country_name', 'Unknown')
            
            print(f"  🏛️  Place: {name}")
            print(f"  🌍 Country: {country}")
            print(f"  📐 Coordinates: ({lat:.4f}, {lon:.4f})")
            print()
        else:
            print(f"  ❌ Location '{toponym}' could not be resolved to coordinates")
            print()

📍 DETAILED LOCATION DATA:

Sentence 1: "I traveled from New York to Richmond, Virginia last summer."
  🏛️  Place: New York
  🌍 Country: United States
  📐 Coordinates: (43.0003, -75.4999)

  🏛️  Place: Richmond
  🌍 Country: United States
  📐 Coordinates: (37.5538, -77.4603)

  🏛️  Place: Virginia
  🌍 Country: United States
  📐 Coordinates: (37.5481, -77.4467)


Sentence 2: "The battle took place near Harrisonburg in the Shenandoah Valley."
  🏛️  Place: City of Harrisonburg
  🌍 Country: United States
  📐 Coordinates: (38.4496, -78.8689)

  🏛️  Place: Community Christian School of the Shenandoah Valley
  🌍 Country: United States
  📐 Coordinates: (38.9104, -78.4778)


Sentence 3: "London and Paris are popular European destinations."
  🏛️  Place: London
  🌍 Country: United Kingdom
  📐 Coordinates: (51.5085, -0.1257)

  🏛️  Place: Paris
  🌍 Country: France
  📐 Coordinates: (48.8534, 2.3488)



## Part 2: Load and Process JMU Text Data

Now we'll want to extract locations from the Reddit data. We'll do so by loading the `.pickle` file from the previous lesson.

### Step 2.1: Load `JMU_reddit.pickle`

This dataset contains sentences from the JMU reddit that have already been tagged for locations.

In [6]:
try:
    df_jmu_reddit_toponyms = pd.read_pickle('data/jmu_reddit_toponyms.pickle')
    print(f"✅ Loaded {len(df_jmu_reddit_toponyms):,} sentences")
   
except FileNotFoundError:
    print("❌ Data file not found!")
    print("You may need to run previous lessons first to generate the sentiment data.")
    print("Or check that you're in the correct directory.")

✅ Loaded 30,005 sentences


### Step 2.2: The Geoparsing Function

Here's a streamlined function that processes text and extracts geographic information:

**Key features:**
- Processes multiple sentences at once for efficiency
- Extracts coordinates and place information

In [7]:
def geoparse_dataframe(df, text_column='sentences'):
    """
    Extract geographic locations from text data.
    
    Args:
        df: DataFrame with text data
        text_column: Column containing the text to parse
    
    Returns:
        DataFrame with added location columns
    """
    print(f"🔍 Processing {len(df)} sentences for geographic locations...")
    
    # Convert text column to list for batch processing
    sentences = df[text_column].tolist()
    
    try:
        # Process all sentences at once (more efficient)
        docs = geo.parse(sentences)  # Parse all sentences without feature filtering
        
        # Initialize storage for results
        places, latitudes, longitudes, feature_names = [], [], [], []
        
        # Extract information from each processed document
        for doc in tqdm(docs, desc="Extracting locations"):
            doc_places = []
            doc_latitudes = []
            doc_longitudes = []
            doc_feature_names = []
            
            # Get all toponyms found in this document
            for toponym in doc.toponyms:
                if toponym.location:
                    doc_places.append(toponym.location.get('name'))
                    doc_latitudes.append(toponym.location.get('latitude'))
                    doc_longitudes.append(toponym.location.get('longitude'))
                    doc_feature_names.append(toponym.location.get('feature_name'))
            
            # Store results (empty lists if no locations found)
            places.append(doc_places)
            latitudes.append(doc_latitudes)
            longitudes.append(doc_longitudes)
            feature_names.append(doc_feature_names)
        
        # Add new columns to dataframe
        df_result = df.copy()
        df_result['place'] = places
        df_result['latitude'] = latitudes
        df_result['longitude'] = longitudes
        df_result['feature_name'] = feature_names
        
        print(f"✅ Geoparsing complete!")
        return df_result
        
    except Exception as e:
        print(f"❌ Error during geoparsing: {e}")
        return df

There are several interesting things of note in the data. First, for some of the sentences the tokenizer did not find a toponym which is indicated by empty lists `[]`. This because this is a more accurate tokenizer and will likely have fewer false positives. We will have to remember to remove these. 

Likewise, right now the parsing has been set to include Administrative areas like countries and states (i.e. The US and Virginia) and population centers (Richmond, Harrisonburg). We will have to think of how to deal with these down the road.

**We can run the geoparser for all the data and expect to wait at least an hour!**

In [8]:
# Run the geoparser over the entire 'sentences' column
geoparse_results = geoparse_dataframe(df_jmu_reddit_toponyms)

🔍 Processing 30005 sentences for geographic locations...
Toponym Recognition...


Batches:   0%|          | 0/30005 [00:00<?, ?it/s]

Toponym Resolution...


Batches:   0%|          | 0/1650 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (849 > 512). Running this sequence through the model will result in indexing errors


Batches:   0%|          | 0/266 [00:00<?, ?it/s]

Extracting locations:   0%|          | 0/30005 [00:00<?, ?it/s]

✅ Geoparsing complete!


In [10]:
# Display the updated DataFrame with new columns
geoparse_results.sample(50)

,type,date,score,year_month,sentences,toponyms,place,latitude,longitude,feature_name
4482,comment,2019-03-30 10:09:43,8,2019-03,So I feel you.,None,[],[],[],[]
5880,comment,2019-03-24 21:31:18,7,2019-03,at least u have time to join a club.,None,[],[],[],[]
5110,comment,2020-07-18 17:16:29,1,2020-07,You can look at the campus map they have.,None,[],[],[],[]
5802,comment,2021-07-09 16:31:59,1,2021-07,Ahh yeah they really should’ve sent out more d...,None,[],[],[],[]
11324,post,2020-05-30 15:41:48,23,2020-05,For meeting people that is.,None,[],[],[],[]
3225,comment,2022-07-16 11:44:15,3,2022-07,> The faculty in my major punished me for viol...,None,[],[],[],[]
1118,comment,2020-08-06 12:21:01,22,2020-08,"As a fellow alum, amen.",None,[],[],[],[]
10089,comment,2020-02-23 15:52:26,7,2020-02,"They will probably be the ones pulled over, wh...",None,[],[],[],[]
7929,comment,2014-03-05 11:09:20,7,2014-03,It's an art.,None,[],[],[],[]
7666,comment,2018-01-14 22:55:10,8,2018-01,"I believe that section is called ""North"" campus.",None,[],[],[],[]


In [11]:
df_reddit_geoparsed = geoparse_results[geoparse_results['place'].apply(len) != 0]


---

In [12]:
df_reddit_geoparsed.sample(50)

,type,date,score,year_month,sentences,toponyms,place,latitude,longitude,feature_name
6312,comment,2024-09-25 12:31:37,11,2024-09,"If you like Mediterranean food, try Xenia.",[Xenia],[Mediterranean],[35.33549],[25.13111],[None]
3814,comment,2020-01-25 18:53:23,9,2020-01,P.S.,[P.S.],[Ṭikar],[27.4813],[83.52467],[None]
1908,comment,2020-07-25 05:01:58,12,2020-07,The Village is slated for demolition and repla...,None,[Village],[51.44978],[-0.08675],[None]
3930,comment,2020-09-24 08:12:36,9,2020-09,No one killed but police violence and harassme...,[Harrisonburg],[City of Harrisonburg],[38.44957],[-78.86892],[None]
5814,comment,2020-07-08 12:38:18,4,2020-07,Old photos of Jackson Hall show the fan window...,None,[Jackson Hall],[38.82861],[-77.30139],[None]
6072,post,2016-12-16 22:19:00,48,2016-12,"Frisco, TX",[Frisco],"[Frisco, Texas]","[33.15067, 31.25044]","[-96.82361, -99.25061]","[None, None]"
5068,post,2021-05-28 22:59:42,59,2021-05,Found in Pheasant Run with no collar,None,[Pheasant Run],[40.31011],[-75.106],[None]
10116,comment,2013-11-13 18:43:49,1,2013-11,86 degrees here in LA.,[LA],[Los Angeles],[34.05223],[-118.24368],[None]
1548,comment,2023-11-26 23:41:15,3,2023-11,ESPN’s latest projections say Gasparilla Bowl ...,[Memphis],[Memphis],[35.14953],[-90.04898],[None]
7845,comment,2020-08-05 09:31:26,11,2020-08,"la morena, mashita, little grill, boboko, indi...",None,"[La Morena, Las Lolas]","[21.23831, 9.51604]","[-86.86751, -72.66671]","[None, None]"


In [13]:
df_reddit_geoparsed_long = df_reddit_geoparsed.explode(['place', 'latitude', 'longitude', 'feature_name']).reset_index(drop=True)
df_reddit_geoparsed_long.sample(50)

,type,date,score,year_month,sentences,toponyms,place,latitude,longitude,feature_name
1065,comment,2021-08-21 20:46:43,18,2021-08,"I've heard 83 is a drug trafficking highway, i...",None,Interchange 83,41.36843,-72.11563,None
908,comment,2022-10-16 09:51:27,17,2022-10,But renaming Ashby to the Harrison isn’t actua...,None,Ashby,45.10012,-77.44948,None
489,comment,2020-03-18 22:28:31,1,2020-03,Are you in Eagle getting high right now?,None,Basalt,39.36887,-107.03282,None
1251,post,2023-04-05 13:07:58,35,2023-04,Skyline dorms are also decently new dorms that...,None,Hubei,31.0,112.25,None
1659,post,2024-07-30 13:59:53,29,2024-07,My cousin who lives in Northern Virginia says ...,None,Nova,64.48632,11.0805,None
88,comment,2019-12-02 13:41:41,5,2019-12,If anyone's ever been over the mountain to Joh...,None,John Paul Jones Arena,38.04583,-78.50722,None
1613,post,2024-11-11 07:52:44,28,2024-11,just applied and wanna know what harrisonburg ...,[harrisonburg],Harrisonburg,38.44957,-78.86892,None
557,comment,2023-11-01 00:48:39,1,2023-11,I recommend the top floor of Roop Hall.,None,Roop Hall,38.4375,-78.87472,None
324,comment,2020-04-14 13:56:28,2,2020-04,North Carolina has had decriminalization for d...,[North Carolina],North Carolina,35.50069,-80.00032,None
1406,comment,2022-07-03 22:53:05,3,2022-07,I'm nowhere near you and I'm not heading to Ha...,[Harrisonburg],City of Harrisonburg,38.44957,-78.86892,None


In [ ]:
df_reddit_geoparsed_long.to_pickle('data/jmu_reddit_geoparsed_long.pickle')


In [2]:
import pandas as pd

df_reddit_geoparsed_long = pd.read_pickle('data/jmu_reddit_geoparsed_long.pickle')
df_reddit_geoparsed_long.to_csv('data/jmu_reddit_geoparsed_long.csv', index=False)

## Part 3: Visualizing Spatial Relations

We can make a quick map of the results using `plotly` and `mapbox`

### Step 3.1: Load Pre-processed Data

The complete geoparsing process took over an hour on a modern computer.

In [15]:
# Count each place while keeping latitude and longitude
df_reddit_geoparsed_count = df_reddit_geoparsed_long.groupby('place').agg({
    'latitude': 'first',    # Keep the first latitude for each place
    'longitude': 'first',   # Keep the first longitude for each place
    'place': 'size'         # Count the occurrences
}).rename(columns={'place': 'count'}).reset_index()

# Filter for places mentioned more than 4 times
df_reddit_geoparsed_count = df_reddit_geoparsed_count[df_reddit_geoparsed_count['count'] > 4]
df_reddit_geoparsed_count.sample(50)

,place,latitude,longitude,count
292,Memorial,29.77303,-95.58442,5
476,Valley View Early College Campus,26.13082,-98.24734,5
261,Lexington,37.98869,-84.47772,5
176,Godwin Hall,36.8543,-82.7584,5
345,Pennsylvania,40.8,-77.7,7
194,Harrisonburg,38.44957,-78.86892,96
321,New York City,40.71427,-74.00597,7
231,Jiamusi,46.79711,130.31118,40
36,Basalt,39.36887,-107.03282,19
326,North Carolina,35.50069,-80.00032,7


In [22]:
fig = px.scatter_map(
    df_reddit_geoparsed_count,     # DataFrame with count data
    lat="latitude",               # Latitude column
    lon="longitude",              # Longitude column
    hover_name="place",           # Show place name on hover
    size="count",                 # Marker size based on count
    hover_data={
        "count": True,              # Show count in hover
        "latitude": False,          # Hide coordinates in hover
        "longitude": False
    },
    title="Geographic Locations Found in JMU Reddit Data",
    zoom=3,                       # Start with a broad view
    height=600
)

# Update the layout to use the default map style
fig.update_layout(
    map_style="carto-voyager",  # No token needed for this style
    margin={"r":0,"t":50,"l":0,"b":0}  # Remove margins but keep space for title
)

fig.show()

#### Critical Questons

- What does the map show about the locations? 
- Where does it do well? 
- Where are the mistakes?